# Étape 1 — Récupérer et contrôler les données

**Projet Streamlit EFREI 2026 : « La France a-t-elle vraiment réduit sa consommation de gaz ? »**

| | |
|---|---|
| Portail | data.gouv.fr |
| Jeu principal | [Consommation quotidienne brute régionale](https://www.data.gouv.fr/datasets/consommation-quotidienne-brute-regionale) |
| Jeu de contrôle | [Consommation quotidienne brute (nationale)](https://www.data.gouv.fr/datasets/consommation-quotidienne-brute) |
| Producteurs | NaTran (ex-GRTgaz), Teréga, RTE, via Open Data Réseaux Énergies (ODRÉ) |
| Licence | Licence Ouverte / Open Licence 2.0 (Etalab) |

**Ce notebook :**
1. télécharge les deux jeux depuis l'API de data.gouv.fr ;
2. contrôle les données avant tout nettoyage (structure, période, manquants, doublons, cohérence régions/national) ;
3. enregistre un rapport `rapport_controle.txt`.

Les fichiers sont créés dans le même dossier que ce notebook. Exécuter les cellules dans l'ordre (*Run All*).

In [1]:
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

DOSSIER = Path.cwd()   # dossier du notebook
API = "https://www.data.gouv.fr/api/1/datasets/{}/"

# nom court : (identifiant du jeu sur data.gouv.fr, fichier local)
SOURCES = {
    "regional": ("consommation-quotidienne-brute-regionale", "conso_regionale.csv"),
    "national": ("consommation-quotidienne-brute", "conso_nationale.csv"),
}

## 1. Téléchargement

Pour chaque jeu, on lit sa fiche sur l'API de data.gouv.fr. On y choisit le fichier CSV le plus récent, puis on le télécharge par morceaux, parce qu'il est volumineux.

**Cache :** si le fichier est déjà présent, on ne le retélécharge pas. La version téléchargée (date, URL, licence) est notée dans `sources.json` pour le README.

In [8]:
def telecharger(slug, nom_fichier):
    """Cherche le CSV le plus récent du jeu sur data.gouv.fr et le télécharge."""
    chemin = DOSSIER / nom_fichier
    if chemin.exists():
        print(f"{nom_fichier} déjà présent, pas de nouveau téléchargement")
        return chemin, None

    fiche = requests.get(API.format(slug), timeout=60).json()
    csvs = [r for r in fiche["resources"] if (r.get("format") or "").lower() == "csv"]
    csvs.sort(key=lambda r: r.get("last_modified") or "", reverse=True)
    ressource = csvs[0]
    print(f"Téléchargement de {nom_fichier} (version du {ressource['last_modified'][:10]})…")

    with requests.get(ressource["url"], stream=True, timeout=300) as r:
        r.raise_for_status()
        taille = 0
        with open(chemin, "wb") as f:
            for morceau in r.iter_content(1024 * 1024):
                f.write(morceau)
                taille += len(morceau)
    print(f"  terminé : {taille / 1e6:.0f} Mo")

    infos = {"jeu": fiche["title"], "page": fiche.get("page"), "url": ressource["url"],
             "version": ressource["last_modified"], "licence": fiche.get("license"),
             "telecharge_le": datetime.now().isoformat(timespec="seconds")}
    return chemin, infos


chemins, infos_sources = {}, {}
for nom, (slug, fichier) in SOURCES.items():
    chemins[nom], infos = telecharger(slug, fichier)
    if infos:
        infos_sources[nom] = infos

if infos_sources:
    (DOSSIER / "sources.json").write_text(json.dumps(infos_sources, ensure_ascii=False, indent=2),
                                          encoding="utf-8")

conso_regionale.csv déjà présent, pas de nouveau téléchargement
conso_nationale.csv déjà présent, pas de nouveau téléchargement


## 2. Lecture des fichiers

Les exports d'ODRÉ utilisent souvent le séparateur `;`. Pour ne pas se tromper, on le détecte à partir de la première ligne.

In [9]:
import re
import unicodedata

OPERATEURS = ["natran", "grtgaz", "terega", "rte"]


def normaliser(texte):
    """Minuscules, sans accents, mots séparés par des _."""
    texte = unicodedata.normalize("NFKD", texte).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", texte.lower()).strip("_")


def nom_court(colonne):
    """Nom court d'une colonne à partir de son libellé d'origine."""
    n = normaliser(colonne)
    op = next((o for o in OPERATEURS if o in n), "")
    if "date" in n and "heure" in n:
        return "date_heure"
    if n in ("date", "heure", "region", "flag_ignore"):
        return n
    if "code" in n and "region" in n:
        return "code_region"
    if "statut" in n:
        return f"statut_{op}"
    if "gaz" in n:
        return "gaz_total_mw" if "totale" in n else f"gaz_{op}_mw"
    if "electricite" in n:
        return f"elec_{op}_mw"
    if "totale" in n:
        return "total_mw"
    return n


def lire_csv(chemin):
    """Lit le CSV (séparateur détecté) et renomme les colonnes."""
    with open(chemin, encoding="utf-8") as f:
        entete = f.readline()
    sep = ";" if entete.count(";") > entete.count(",") else ","
    df = pd.read_csv(chemin, sep=sep, low_memory=False)
    correspondance = {c: nom_court(c) for c in df.columns}
    return df.rename(columns=correspondance), correspondance


reg, noms_reg = lire_csv(chemins["regional"])
nat, noms_nat = lire_csv(chemins["national"])
print(f"Régional : {reg.shape[0]:,} lignes x {reg.shape[1]} colonnes")
print(f"National : {nat.shape[0]:,} lignes x {nat.shape[1]} colonnes")

print("\nCorrespondance des noms de colonnes (jeu régional) :")
print(pd.Series(noms_reg, name="nom court").rename_axis("libellé d'origine").to_string())

Régional : 2,874,807 lignes x 14 colonnes
National : 257,136 lignes x 12 colonnes

Correspondance des noms de colonnes (jeu régional) :
libellé d'origine
Date - Heure                                       date_heure
Date                                                     date
Heure                                                   heure
Code INSEE région                                 code_region
Région                                                 region
Consommation brute gaz (MW PCS 0°C) - NaTran    gaz_natran_mw
Statut - NaTran                                 statut_natran
Consommation brute gaz (MW PCS 0°C) - Teréga    gaz_terega_mw
Statut - Teréga                                 statut_terega
Consommation brute gaz totale (MW PCS 0°C)       gaz_total_mw
Consommation brute électricité (MW) - RTE         elec_rte_mw
Statut - RTE                                       statut_rte
Consommation brute totale (MW)                       total_mw
flag_ignore                             

## 3. Contrôle des données

Pour chaque jeu, la fonction ci-dessous affiche :
- les colonnes et leurs types, et un aperçu des premières lignes ;
- la période couverte, et le nombre de valeurs de gaz renseignées par année (pour savoir à partir de quand la série est utilisable) ;
- le nombre de lignes par région ;
- le pourcentage de valeurs manquantes par colonne ;
- les doublons (même date et heure, même région) ;
- les statuts des données (provisoire, consolidé, définitif) ;
- un résumé des variables numériques (minimum, médiane, maximum, valeurs négatives).

Tout ce qui est affiché est aussi gardé pour le rapport final.

In [10]:
rapport = []

def ecrire(texte=""):
    """Affiche le texte et le garde pour le rapport."""
    print(texte)
    rapport.append(str(texte))


def controler(nom, df):
    ecrire(f"\n{'=' * 60}\n{nom.upper()} : {len(df):,} lignes x {df.shape[1]} colonnes\n{'=' * 60}")

    ecrire("\n--- Colonnes et types ---")
    ecrire(df.dtypes.to_string())

    ecrire("\n--- Aperçu ---")
    ecrire(df.head(4).to_string())

    if "date" in df.columns:
        dates = pd.to_datetime(df["date"], errors="coerce")
        ecrire(f"\n--- Période : du {dates.min():%Y-%m-%d} au {dates.max():%Y-%m-%d} ---")
        gaz = [c for c in df.columns if "gaz" in c and "statut" not in c]
        par_an = df.groupby(dates.dt.year).agg(lignes=("date", "size"),
                                               **{c: (c, "count") for c in gaz})
        ecrire("Lignes et valeurs de gaz renseignées par année :")
        ecrire(par_an.to_string())

    if "region" in df.columns:
        ecrire("\n--- Régions (nombre de lignes) ---")
        ecrire(df["region"].value_counts().to_string())

    ecrire("\n--- Valeurs manquantes ---")
    manquants = pd.DataFrame({"nombre": df.isna().sum(), "pct": (df.isna().mean() * 100).round(1)})
    ecrire(manquants.sort_values("nombre", ascending=False).to_string())

    # Le gaz est horaire mais le fichier est demi-horaire : on regarde les heures pleines seulement
    if "heure" in df.columns:
        heures_pleines = df[df["heure"].astype(str).str.endswith(":00")]
        gaz = [c for c in df.columns
               if c.startswith(("gaz_", "statut_natran", "statut_grtgaz", "statut_terega"))]
        ecrire(f"\n--- Gaz manquant aux heures pleines ({len(heures_pleines):,} lignes) ---")
        ecrire(pd.DataFrame({"nombre": heures_pleines[gaz].isna().sum(),
                             "pct": (heures_pleines[gaz].isna().mean() * 100).round(1)}).to_string())
        if "region" in df.columns and "gaz_terega_mw" in df.columns:
            ecrire("\nPart des heures pleines avec une valeur Teréga, par région (%) :")
            ecrire((heures_pleines.groupby("region")["gaz_terega_mw"].count()
                    / heures_pleines.groupby("region").size() * 100).round(1).to_string())

    ecrire("\n--- Doublons ---")
    cle = [c for c in ["date_heure", "region"] if c in df.columns]
    if cle:
        ecrire(f"Lignes en double sur {cle} : {df.duplicated(subset=cle).sum()}")
    ecrire(f"Lignes entièrement identiques : {df.duplicated().sum()}")

    for c in [c for c in df.columns if c.startswith(("statut", "flag"))]:
        ecrire(f"\n--- Valeurs de {c} ---")
        ecrire(df[c].value_counts(dropna=False).to_string())

    ecrire("\n--- Variables numériques ---")
    num = df.select_dtypes("number").drop(columns=["code_region", "flag_ignore"], errors="ignore")
    resume = num.describe().T[["count", "min", "50%", "max"]]
    resume["negatifs"] = (num < 0).sum()
    ecrire(resume.round(1).to_string())

### 3.1 Jeu régional

In [17]:
controler("regional", reg)


REGIONAL : 2,874,807 lignes x 14 colonnes

--- Colonnes et types ---
date_heure           str
date                 str
heure                str
code_region        int64
region               str
gaz_natran_mw    float64
statut_natran        str
gaz_terega_mw    float64
statut_terega        str
gaz_total_mw     float64
elec_rte_mw      float64
statut_rte           str
total_mw         float64
flag_ignore          str

--- Aperçu ---
                  date_heure        date  heure  code_region                region  gaz_natran_mw statut_natran  gaz_terega_mw statut_terega  gaz_total_mw  elec_rte_mw statut_rte  total_mw flag_ignore
0  2012-12-31T23:00:00+00:00  2013-01-01  00:00           76             Occitanie            NaN           NaN         1945.0     Définitif           NaN          NaN        NaN       NaN         non
1  2012-12-31T23:00:00+00:00  2013-01-01  00:00           84  Auvergne-Rhône-Alpes            NaN           NaN           34.0     Définitif           NaN        

### 3.2 Jeu national

In [18]:
controler("national", nat)


NATIONAL : 257,136 lignes x 12 colonnes

--- Colonnes et types ---
date_heure           str
date                 str
heure                str
gaz_natran_mw    float64
statut_natran        str
gaz_terega_mw    float64
statut_terega        str
gaz_total_mw     float64
elec_rte_mw        int64
statut_rte           str
total_mw         float64
flag_ignore          str

--- Aperçu ---
                  date_heure        date  heure  gaz_natran_mw statut_natran  gaz_terega_mw  statut_terega  gaz_total_mw  elec_rte_mw statut_rte  total_mw flag_ignore
0  2026-08-31T21:30:00+00:00  31/08/2026  23:30            NaN           NaN            NaN            NaN           NaN        43265  Consolidé       NaN         non
1  2026-08-31T21:00:00+00:00  31/08/2026  23:00        20075.0     Définitif          886.0  Intermédiaire       20961.0        43979  Consolidé   64940.0         non
2  2026-08-31T20:30:00+00:00  31/08/2026  22:30            NaN           NaN            NaN            NaN         

C:\Users\DELL\AppData\Local\Temp\ipykernel_10928\2120910176.py:19: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dates = pd.to_datetime(df["date"], errors="coerce")


Lignes et valeurs de gaz renseignées par année :
      lignes  gaz_natran_mw  gaz_terega_mw  gaz_total_mw
date                                                    
2012   17568           8784           8784          8784
2013   17520           8760           8760          8760
2014   17520           8760           8760          8760
2015   17520           8760           8760          8760
2016   17568           8784           8784          8784
2017   17520           8760           8760          8760
2018   17520           8760           8760          8760
2019   17520           8760           8760          8760
2020   17568           8784           8784          8783
2021   17520           8760           8760          8760
2022   17520           8760           8757          8757
2023   17520           8760           8760          8760
2024   17568           8784           8784          8784
2025   17520           8760           8760          8760
2026   11664           5832           5

## 4. Validation croisée : somme des régions = national ?

Chaque jour, la somme des régions devrait être proche du total national. Un écart régulier peut avoir plusieurs causes : un périmètre différent (Corse, consommations non affectées à une région), des jours incomplets ou des données provisoires. On le documentera dans la section « Qualité des données » de l'app.

In [15]:
def en_date(serie):
    """Convertit les dates, qu'elles soient au format 2013-01-31 ou 31/01/2013."""
    iso = pd.to_datetime(serie, format="%Y-%m-%d", errors="coerce")
    fr = pd.to_datetime(serie, format="%d/%m/%Y", errors="coerce")
    return iso.fillna(fr)


communes = [c for c in reg.select_dtypes("number").columns
            if c in nat.columns and c.endswith("_mw")]

somme_regions = reg.assign(jour=en_date(reg["date"])).groupby("jour")[communes].sum(min_count=1)
national = nat.assign(jour=en_date(nat["date"])).groupby("jour")[communes].sum(min_count=1)
ecart = ((somme_regions - national) / national * 100).dropna(how="all")

ecrire(f"\n{'=' * 60}\nVALIDATION : somme des régions vs national (par jour)\n{'=' * 60}")
ecrire(f"{len(ecart)} jours comparés. Écart en % :")
ecrire(ecart.describe().T[["50%", "min", "max"]].round(2).to_string())
ecrire("\nÉcart médian par année (%) :")
ecrire(ecart.groupby(ecart.index.year)[["gaz_total_mw", "elec_rte_mw"]].median().round(2).to_string())


VALIDATION : somme des régions vs national (par jour)
4991 jours comparés. Écart en % :
               50%    min     max
gaz_natran_mw -0.0 -68.46   47.25
gaz_terega_mw  0.0 -28.33  366.04
gaz_total_mw  -0.0 -64.16   44.92
elec_rte_mw    0.0  -2.70    3.65
total_mw      -0.0 -36.15   22.11

Écart médian par année (%) :
      gaz_total_mw  elec_rte_mw
jour                           
2013         -0.06        -0.02
2014         -0.00         0.00
2015         -0.00         0.00
2016          0.00         0.00
2017          0.00         0.00
2018          0.00         0.00
2019          0.00         0.00
2020         -0.00         0.00
2021          0.04         0.00
2022          0.05        -0.03
2023         -0.04         0.00
2024         -0.02        -0.50
2025         -0.00         0.00
2026         -0.00         0.00


In [16]:
# Examen des anomalies repérées
doublons = reg[reg.duplicated(subset=["date_heure", "region"], keep=False)]
ecrire("\n--- Jours concernés par les doublons (régional) ---")
ecrire(doublons["date"].value_counts().sort_index().to_string())
ecrire("\nExemple :")
ecrire(doublons.sort_values(["region", "date_heure"]).head(6).to_string())

ecrire("\n--- Électricité négative ---")
ecrire(reg[reg["elec_rte_mw"] < 0].to_string())

ecrire("\n--- Statut Teréga « vers » ---")
ecrire(reg[reg["statut_terega"] == "vers"].to_string())

ecrire("\n--- Année des données NaTran « Redressé » ---")
ecrire(reg.loc[reg["statut_natran"] == "Redressé", "date"].str[:4].value_counts().to_string())


--- Jours concernés par les doublons (régional) ---
date
2013-03-31    48
2014-03-30    48
2015-03-29    48
2016-03-27    48
2017-03-26    48
2018-03-25    48
2019-03-31    48
2020-03-29    48
2021-03-28    48
2022-03-27    48
2023-03-26    48
2024-03-31    48
2025-03-30    48
2026-03-29    48

Exemple :
                       date_heure        date  heure  code_region                region  gaz_natran_mw statut_natran  gaz_terega_mw statut_terega  gaz_total_mw  elec_rte_mw statut_rte  total_mw flag_ignore
51321   2013-03-31T01:00:00+00:00  2013-03-31  03:00           84  Auvergne-Rhône-Alpes         7109.0     Définitif           37.0     Définitif        7146.0       7809.0  Définitif   14955.0         non
51323   2013-03-31T01:00:00+00:00  2013-03-31  02:00           84  Auvergne-Rhône-Alpes         7056.0     Définitif           34.0     Définitif        7090.0       7809.0  Définitif   14899.0         non
51329   2013-03-31T01:30:00+00:00  2013-03-31  02:30           84  Auvergne

## 5. Enregistrement du rapport

In [19]:
(DOSSIER / "rapport_controle.txt").write_text("\n".join(rapport), encoding="utf-8")
print("Rapport enregistré : rapport_controle.txt")

Rapport enregistré : rapport_controle.txt
